# EtherLens — live analysis on the Dataproc cluster

This notebook runs on the cluster's **PySpark** kernel (`spark` is already created).
It reads the silver and gold tables the batch jobs wrote to Cloud Storage and runs
a few Spark SQL queries live, to show that the analysis really runs on the cluster.

Data: Ethereum mainnet transactions, January 2025 (~37 M rows after cleaning).

In [ ]:
BUCKET = "etherlens-parvmittal"          # change if your bucket name differs
SILVER = f"gs://{BUCKET}/silver/tx"
GOLD = f"gs://{BUCKET}/gold"

silver = spark.read.parquet(SILVER)
silver.createOrReplaceTempView("silver_tx")
print("silver rows:", silver.count())
silver.printSchema()

## 1. Daily activity (Spark SQL)
`dt` is the partition column, so each day is one folder (one Parquet file) in the bucket.

In [ ]:
spark.sql("""
    SELECT dt, COUNT(*) AS tx, COUNT(DISTINCT from_address) AS senders,
           ROUND(SUM(value_eth), 0) AS eth_moved
    FROM silver_tx GROUP BY dt ORDER BY dt
""").show(31)

## 2. Partition pruning
Filtering on `dt` lets Spark read only that day's file — see `PartitionFilters` in the plan.

In [ ]:
spark.sql("SELECT COUNT(*) FROM silver_tx WHERE dt = '2025-01-15'").explain()
spark.sql("SELECT COUNT(*) AS tx_on_jan_15 FROM silver_tx WHERE dt = '2025-01-15'").show()

## 3. When is gas cheapest? (median gas price by hour of day, UTC)

In [ ]:
import matplotlib.pyplot as plt

by_hour = spark.sql("""
    SELECT hour, percentile_approx(gas_price_gwei, 0.5) AS median_gwei, COUNT(*) AS tx
    FROM silver_tx GROUP BY hour ORDER BY hour
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_hour["hour"], by_hour["median_gwei"])
ax.set_xlabel("hour of day (UTC)"); ax.set_ylabel("median gas price (gwei)")
ax.set_title("Median gas price by hour, January 2025")
plt.show()
by_hour.sort_values("median_gwei").head(3)

## 4. Most central addresses (PageRank, last 7 days)

In [ ]:
spark.read.parquet(f"{GOLD}/pagerank").orderBy("rank", ascending=False).show(10, truncate=False)

## 5. Anomaly detection (KMeans): cluster sizes and flagged addresses

In [ ]:
clusters = spark.read.parquet(f"{GOLD}/address_clusters")
clusters.groupBy("cluster").count().orderBy("cluster").show()
flagged = spark.read.parquet(f"{GOLD}/flagged_addresses")
flagged.groupBy("reason").count().show()
flagged.orderBy("score", ascending=False).show(5, truncate=False)